# TRON Autopilot CUDA proof — pythia-1.4b

Runtime → GPU (T4) → Run all. Uses Offload when the advisor says the model needs it.

In [ ]:
import os, pathlib, subprocess, sys
assert __import__('torch').cuda.is_available(), 'Enable GPU runtime first'
print('GPU:', __import__('torch').cuda.get_device_name(0))
REPO = '/content/tron-client'
BRANCH = 'cursor/economic-autopilot-38d1'
if not pathlib.Path(REPO, '.git').exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, 'https://github.com/StarkX-cloud/tron-client.git', REPO])
os.chdir(REPO)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-dev.txt', '-r', 'requirements-training.txt'])
out = '/content/economic_autopilot_cuda_1_4b_benchmark.json'
subprocess.check_call([
    sys.executable, '-m', 'benchmarks.economic_autopilot',
    '--real-model', '--trials', '1',
    '--model', 'EleutherAI/pythia-1.4b',
    '--max-new-tokens', '8',
    '--hourly-rate', '0.50',
    '--out', out,
])
subprocess.call(['nvidia-smi'])
print(pathlib.Path(out).read_text()[:3000])
try:
    from google.colab import files
    files.download(out)
except Exception:
    print('Download:', out)